In [ ]:
import os
import datetime
from functools import partial

import colormaps
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import xarray as xr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import MaxNLocator
from pathlib import Path
from tqdm import tqdm
from jetutils.anyspell import get_persistent_jet_spells, mask_from_spells_pl, subset_around_onset, get_persistent_spell_times_from_som, get_spells, extend_spells, gb_index, make_daily
from jetutils.clustering import Experiment, RAW_REALSPACE, labels_to_centers
from jetutils.data import standardize, extract
from jetutils.definitions import DATADIR, KAPPA
from jetutils.jet_finding import gaussian_smooth_func, do_everything
import polars.selectors as cs

os.environ["RUST_BACKTRACE"] = "1"

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline

In [ ]:
ds: xr.Dataset = standardize(xr.open_dataset("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/uv/6H/195902.nc"))
ds.to_zarr("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/uv/6H/full.zarr", append_dim="time", mode="a")

In [ ]:
import xarray as xr
xr.open_zarr("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/uv/6H/full.zarr").to

In [ ]:
ds = xr.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3',
    chunks=None,
    storage_options=dict(token='anon'),
)
vars = ["v_component_of_wind", "u_component_of_wind", "vertical_velocity", "temperature"]
levs = [175, 200, 225, 250, 300, 350]
time_filter = (
    np.isin(ds.time.dt.year, np.arange(1959, 2025))
    & (ds.time.dt.hour % 6 == 0)
)
ds = standardize(ds.isel(time=time_filter)[vars]).sel(lev=levs).sel(lon=np.arange(-80, 40.5, 0.5), lat=np.arange(15, 80.5, 0.5))

In [ ]:
to_comp = ds.drop_encoding().to_zarr("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/uv/6H/full.zarr", consolidated=False, compute=False, mode="w")

In [ ]:
from dask.diagnostics.progress import ProgressBar
with ProgressBar():
    to_comp.compute()

# rest

In [ ]:
kwargs = dict(
    n_coarsen=1,
    base_s_thresh=0.55,
    alignment_thresh=0.6,
    int_thresh_factor=0.6,
    hole_size=6,
    smooth_func=partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.8),
)

both_jets = {}
both_paths = {}
for run in ["historical", "ssp370"]:
    path = Path(DATADIR, "CESM2/high_wind", run)
    ds = xr.open_dataset(path.joinpath("ds.zarr"))
    ds = standardize(ds)
    ds["theta"] = ds["t"] * (1000 / ds["lev"]) ** KAPPA
    ds = extract(
        ds, minlon=-80, maxlon=40, minlat=15, maxlat=80
    )
    jets, ph_jets, props, props_full = do_everything(ds, path.joinpath("results/1"), feature_names=("s", "theta"), n_init=3, **kwargs)
    both_paths[run] = path
    both_jets[run] = ph_jets